# Epic: Did that lemma come from speech or narrative?

Given a lemma, what is the likelihood that it came from direct speech versus narrative?

In [2]:
import numpy as np
import pandas as pd

narrative_df = pd.read_parquet("../parquet/homeric_narrative_dtm.parquet")
speeches_df = pd.read_parquet("../parquet/homeric_speeches_dtm.parquet")

In [3]:
n_lemmata_iliad_narrative = narrative_df["Iliad"].sum()
n_lemmata_odyssey_narrative = narrative_df["Odyssey"].sum()

print(f"Number of unique (non-stopword) lemmata in Iliadic narrative: {n_lemmata_iliad_narrative}")
print(f"Number of unique (non-stopword) lemmata in Odyssean narrative: {n_lemmata_odyssey_narrative}")
print("\n\n")

iliad_narrative_pcts = narrative_df["Iliad"] / n_lemmata_iliad_narrative
odyssey_narrative_pcts = narrative_df["Odyssey"] / n_lemmata_odyssey_narrative

print(iliad_narrative_pcts.sort_values(ascending=False).head(100))
iliad_narrative_pcts.sort_values(ascending=False).head(100).to_csv("../csv/iliad_narrative_top_100_lemmata.csv")
print("\n\n")
print(odyssey_narrative_pcts.sort_values(ascending=False).head(100))
odyssey_narrative_pcts.sort_values(ascending=False).head(100).to_csv("../csv/odyssey_narrative_top_100_lemmata.csv")

print("\n\n")

total_lemmata_narrative = n_lemmata_iliad_narrative + n_lemmata_odyssey_narrative
narrative_df["all"] = narrative_df["Iliad"] + narrative_df["Odyssey"]
all_pcts = narrative_df["all"] / total_lemmata_narrative

all_pcts.sort_values(ascending=False).head(100).to_csv("../csv/all_narrative_top_100_lemmata.csv")


Number of unique (non-stopword) lemmata in Iliadic narrative: 21455
Number of unique (non-stopword) lemmata in Odyssean narrative: 9646



lemma
ναῦς         0.008203
Ἕκτωρ        0.008110
υἱός         0.008063
Τρώς         0.007551
Ἀχαιός       0.007318
               ...   
Ἀντίλοχος    0.001585
ἀγαθός       0.001585
μακρός       0.001585
ὄρος         0.001585
αἷμα         0.001585
Name: Iliad, Length: 100, dtype: float64



lemma
Ὀδυσσεύς     0.019075
Τηλέμαχος    0.008501
χείρ         0.007983
βαίνω        0.007775
Ἀθήνη        0.007361
               ...   
αὐτίκος      0.001555
χέω          0.001555
κακός        0.001555
Μενέλαος     0.001555
δαίς         0.001555
Name: Odyssey, Length: 100, dtype: float64





In [4]:
n_lemmata_speeches = speeches_df.sum(axis=1).sum()
speeches_pcts = speeches_df.sum(axis=1) / n_lemmata_speeches

speeches_pcts.sort_values(ascending=False).head(100).to_csv("../csv/all_speeches_top_100_lemmata.csv")

In [5]:
n_lemmata_iliad_speeches = speeches_df["Iliad"].sum(axis=1).sum()
n_lemmata_odyssey_speeches = speeches_df["Odyssey"].sum(axis=1).sum()

iliad_speeches_pcts = speeches_df["Iliad"].sum(axis=1) / n_lemmata_iliad_speeches
odyssey_speeches_pcts = speeches_df["Odyssey"].sum(axis=1) / n_lemmata_odyssey_speeches

iliad_speeches_pcts.sort_values(ascending=False).head(100).to_csv("../csv/iliad_speeches_top_100_lemmata.csv")
odyssey_speeches_pcts.sort_values(ascending=False).head(100).to_csv("../csv/odyssey_speeches_top_100_lemmata.csv")


## Speech versus narrative 

In [6]:
df = pd.read_parquet("../parquet/homer_speech_narrative.parquet")

def build_totals(df: pd.DataFrame) -> dict[tuple[str, str], int]:
    return {key: int(df[key].sum()) for key in df.columns}

totals = build_totals(df)

In [7]:
def log_likelihood(a: pd.Series, b: pd.Series, total_a: int, total_b: int) -> pd.Series:
    """Dunning G² for each lemma. Positive = skewed toward corpus A."""
    N = total_a + total_b
    expected_a = (a + b) * total_a / N
    expected_b = (a + b) * total_b / N

    def safe_term(obs, exp):
        return np.where(obs > 0, obs * np.log(obs / exp), 0)

    g2 = 2 * (safe_term(a, expected_a) + safe_term(b, expected_b))
    return pd.Series(np.where(a / total_a >= b / total_b, g2, -g2), index=a.index)

In [8]:
totals_iliad = totals["iliad", "speech"] + totals["iliad", "narrative"]
totals_odyssey = totals["odyssey", "speech"] + totals["odyssey", "narrative"]

In [9]:
il = df["iliad"].sum(axis=1)
od = df["odyssey"].sum(axis=1)

loglikelihood_work = log_likelihood(il, od, totals_iliad, totals_odyssey)

/Users/pletcher/code/writing/articles/2026-06-13_ccc-2026/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/pletcher/code/writing/articles/2026-06-13_ccc-2026/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [10]:
loglikelihood_work.sort_values(ascending=False).to_csv("../csv/iliad_vs_odyssey.csv")

In [11]:
import altair as alt

In [12]:
speech = df.xs("speech", axis=1, level="register").sum(axis=1)
narrative = df.xs("narrative", axis=1, level="register").sum(axis=1)

totals_speech = totals["iliad", "speech"] + totals["odyssey", "speech"]
totals_narrative = totals["iliad", "narrative"] + totals["odyssey", "narrative"]

loglikelihood_register = log_likelihood(speech, narrative, totals_speech, totals_narrative)

/Users/pletcher/code/writing/articles/2026-06-13_ccc-2026/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [13]:
loglikelihood_register.sort_values(ascending=False).to_csv("../csv/speech_vs_narrative.csv")

## Plotting lemmata by work

In [14]:
from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests

total_tokens = totals_iliad + totals_odyssey
overall_freq = (il + od) / total_tokens

work_plot_df = pd.DataFrame({
    "lemma": loglikelihood_work.index,
    "log_likelihood": loglikelihood_work.values,
    "relative_frequency": overall_freq.values,
    "work": np.where(loglikelihood_work.values >= 0, "Iliad", "Odyssey"),
})

# p-value from χ^2(df=1); use abs() because G^2 is signed here
work_plot_df["p_value"] = chi2.sf(work_plot_df["log_likelihood"].abs(), df=1)

# Benjamini-Hochberg FDR correction across all lemmata
_, work_plot_df["p_value_fdr"], _, _ = multipletests(work_plot_df["p_value"], method="fdr_bh")

work_plot_df = work_plot_df[work_plot_df["log_likelihood"].abs() >= 30]
print(f"{len(work_plot_df)} lemmata after filtering")
work_plot_df.sort_values("log_likelihood", ascending=False).head(10)

136 lemmata after filtering


,lemma,log_likelihood,relative_frequency,work,p_value,p_value_fdr
7415,Ἕκτωρ,487.914695,0.004147,Iliad,4.050910e-108,1.704623e-104
867,Τρώς,454.562629,0.005939,Iliad,7.331154e-101,2.056633e-97
6426,Ἀχιλλεύς,299.631380,0.003594,Iliad,3.963524e-67,5.559503e-64
7680,ἵππος,251.448289,0.004307,Iliad,1.255172e-56,1.320441e-53
6424,Ἀχαιός,221.224776,0.006746,Iliad,4.889134e-50,4.571884e-47
38,Αἴας,158.489189,0.001877,Iliad,2.419647e-36,1.851250e-33
4155,πόλεμος,147.080645,0.002618,Iliad,7.535323e-34,5.284773e-31
595,Πάτροκλος,137.445629,0.001464,Iliad,9.634544e-32,5.405622e-29
2848,μάχομαι,133.732071,0.002318,Iliad,6.252925e-31,3.289039e-28
2847,μάχη,121.026259,0.001257,Iliad,3.771073e-28,1.763186e-25


In [15]:
alt.Chart(work_plot_df).mark_point(opacity=0.5, size=30).encode(
    x=alt.X("log_likelihood:Q", title="Log-likelihood (← Odyssey | Iliad →)"),
    y=alt.Y(
        "relative_frequency:Q",
        title="Overall relative frequency",
        scale=alt.Scale(type="log"),
    ),
    color=alt.Color(
        "work:N",
        scale=alt.Scale(domain=["Iliad", "Odyssey"], range=["tomato", "steelblue"]),
        legend=alt.Legend(title="More common in"),
    ),
    tooltip=["lemma:N", "log_likelihood:Q", "relative_frequency:Q"],
).properties(
    title="Iliad vs. Odyssey: lemma distinctiveness vs. overall frequency",
    width=650,
    height=450,
)

alt.Chart(...)

## Plotting lemmata by register

In [18]:
total_tokens = totals_narrative + totals_speech
overall_freq = (narrative + speech) / total_tokens

register_plot_df = pd.DataFrame({
    "lemma": loglikelihood_register.index,
    "log_likelihood": loglikelihood_register.values,
    "relative_frequency": overall_freq.values,
    "register": np.where(loglikelihood_register.values >= 0, "Speech", "Narrative"),
})

# p-value from χ^2(df=1); use abs() because G^2 is signed here
register_plot_df["p_value"] = chi2.sf(register_plot_df["log_likelihood"].abs(), df=1)

# Benjamini-Hochberg FDR correction across all lemmata
_, register_plot_df["p_value_fdr"], _, _ = multipletests(register_plot_df["p_value"], method="fdr_bh")

register_plot_df = register_plot_df[register_plot_df["log_likelihood"].abs() >= 10]
print(f"{len(register_plot_df)} lemmata after filtering")
register_plot_df.sort_values("log_likelihood", ascending=False).head(10)

643 lemmata after filtering


,lemma,log_likelihood,relative_frequency,register,p_value,p_value_fdr
2152,κακός,221.019278,0.004110,Speech,5.420700e-50,2.281030e-46
3295,ξένος,173.451491,0.001999,Speech,1.304276e-39,2.744197e-36
3362,οἶδα,159.147268,0.002852,Speech,1.737653e-36,1.828011e-33
1985,θεός,154.899570,0.007215,Speech,1.472790e-35,1.377223e-32
6503,ἐθέλω,147.332802,0.002749,Speech,6.637120e-34,5.585801e-31
1352,δίδωμι,141.984769,0.004541,Speech,9.799674e-33,6.344158e-30
3359,οἴομαι,117.899017,0.001192,Speech,1.824442e-27,1.096750e-24
6128,ἄνθρωπος,116.618181,0.001783,Speech,3.480112e-27,1.952575e-24
4759,φίλος,107.797255,0.006371,Speech,2.977285e-25,1.566052e-22
5474,ἀνήρ,89.178583,0.009749,Speech,3.607333e-21,1.597858e-18


In [19]:
alt.Chart(register_plot_df).mark_point(opacity=0.5, size=30).encode(
    x=alt.X("log_likelihood:Q", title="Log-likelihood (← Narrative | Speech →)"),
    y=alt.Y(
        "relative_frequency:Q",
        title="Overall relative frequency",
        scale=alt.Scale(type="log"),
    ),
    color=alt.Color(
        "register:N",
        scale=alt.Scale(domain=["Speech", "Narrative"], range=["tomato", "steelblue"]),
        legend=alt.Legend(title="More common in"),
    ),
    tooltip=["lemma:N", "log_likelihood:Q", "relative_frequency:Q"],
).properties(
    title="Speech vs. Narrative: lemma distinctiveness vs. overall frequency",
    width=650,
    height=450,
)

alt.Chart(...)